# Lesson 3: From token streams to training batches

Lesson 2 turned documents into GPT-2 token IDs. Here we prepare those IDs for next-token training: separate training and validation documents, concatenate each split independently, then make input and target matrices from a sliding window.

The Python notebook connects the real sample text to token IDs. C and Java use the same small, fixed integer fixture to focus on batching without repeating tokenizer setup. Their batch shapes and one-token shift match the Python fixture exactly.

No PyTorch, model weights, training loop, dataset download, or GPU is needed. The model later receives `x` with shape `(B, T)` and is trained to predict `y` at each position.

**Read the companions:** [C batching_lesson.c](../../c/src/batching_lesson.c) and [Java BatchingLesson.java](../../java/src/main/java/learning/karpathy/gpt2/BatchingLesson.java). Run them from the repository root with `-Action lesson -Lesson 3`.

Reference: Karpathy's [`fineweb.py`](../../build-nanogpt/fineweb.py) creates token shards, and [`train_gpt2.py`](../../build-nanogpt/train_gpt2.py) contains `DataLoaderLite` and the train/validation loader setup.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "gpt-2", *Path.cwd().parents]
tutorial_root = next(
    (candidate for candidate in candidates
     if (candidate / "experiments/configs/tiny-playground.json").is_file()),
    None,
)
if tutorial_root is None:
    raise FileNotFoundError("Open the repository, gpt-2 folder, or its notebook folder and restart the kernel.")

import tiktoken

encoding = tiktoken.get_encoding("gpt2")
end_of_text_id = encoding.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]
sample_path = tutorial_root / "data" / "samples" / "playground.txt"
documents = [line for line in sample_path.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Kernel: {sys.executable}")
print(f"Sample: {sample_path.relative_to(tutorial_root)} ({len(documents)} short documents)")
print(f"GPT-2 end-of-text ID: {end_of_text_id}")


## 1. Split documents before making token streams

For this tiny example, the first half of the documents is training data and the second half is validation data. Real projects choose the split to match their dataset and evaluation goal. Splitting whole documents first prevents a document from appearing in both sets.

Each document starts with GPT-2's end-of-text ID (`50256`). We flatten the documents **within each split**, so no training batch can accidentally continue into validation data.

Karpathy's reference preprocessing follows a different concrete convention: `fineweb.py` writes tokenized shards and marks the first shard as validation; the remaining shards are training. `DataLoaderLite` then reads train and validation shard files separately. Our document split is a small teaching example, not a reproduction of FineWeb's dataset preparation.


In [ ]:
split_at = len(documents) // 2
train_documents = documents[:split_at]
validation_documents = documents[split_at:]

# Prepend one boundary marker, then encode ordinary text with the GPT-2 vocabulary.
def encode_split(document_texts: list[str]) -> list[int]:
    token_ids: list[int] = []
    for document_text in document_texts:
        token_ids.append(end_of_text_id)
        token_ids.extend(encoding.encode_ordinary(document_text))
    return token_ids

train_tokens = encode_split(train_documents)
validation_tokens = encode_split(validation_documents)

print(f"Train:      {len(train_documents)} documents, {len(train_tokens)} token IDs")
print(f"Validation: {len(validation_documents)} documents, {len(validation_tokens)} token IDs")
print(f"Train starts:      {train_tokens[:12]}")
print(f"Validation starts: {validation_tokens[:12]}")
assert set(train_documents).isdisjoint(validation_documents)


## 2. Shift a stream to create inputs and targets

Let `B` be the batch size and `T` the number of tokens per training sequence. One batch needs `B*T + 1` consecutive IDs: the extra ID supplies the final next-token target. We reshape the first `B*T` IDs into `x`, then reshape the same window shifted by one position into `y`.

For `B=2`, `T=4`, the shared fixture is `[101, 102, ..., 109]`:

```text
x = [[101, 102, 103, 104], [105, 106, 107, 108]]
y = [[102, 103, 104, 105], [106, 107, 108, 109]]
```

The second row continues the same stream. In a tensor library these nested lists become integer tensors, often with shape `(B, T)`.


In [ ]:
BATCH_SIZE = 2
CONTEXT_LENGTH = 4


def make_batch(token_ids: list[int], start: int = 0,
               batch_size: int = BATCH_SIZE,
               context_length: int = CONTEXT_LENGTH) -> tuple[list[list[int]], list[list[int]]]:
    # Return B rows of T inputs and their one-token-ahead targets.
    window_size = batch_size * context_length
    if start < 0 or start + window_size + 1 > len(token_ids):
        raise ValueError("The token stream does not contain a complete B*T+1 window.")

    window = token_ids[start:start + window_size + 1]
    inputs = [window[row * context_length:(row + 1) * context_length]
              for row in range(batch_size)]
    shifted = window[1:]
    targets = [shifted[row * context_length:(row + 1) * context_length]
               for row in range(batch_size)]
    return inputs, targets


# This fixed fixture is shared with the C and Java companions.
fixture = list(range(101, 110))
fixture_x, fixture_y = make_batch(fixture)
print("Shared fixture")
print("x =", fixture_x)
print("y =", fixture_y)
assert fixture_x == [[101, 102, 103, 104], [105, 106, 107, 108]]
assert fixture_y == [[102, 103, 104, 105], [106, 107, 108, 109]]

# Apply the same function to actual GPT-2 IDs from the sample corpus.
train_x, train_y = make_batch(train_tokens)
validation_x, validation_y = make_batch(validation_tokens)
print("\nFirst real training batch:")
print("x =", train_x)
print("y =", train_y)
print("\nFirst real validation batch (built from its own stream):")
print("x =", validation_x)
print("y =", validation_y)


## 3. Advance the cursor and keep the split boundary

A loader remembers where the next batch begins. In the single-process toy version below, each call advances by `B*T`. If another full `B*T+1` window would run past the end of this stream, it wraps to the beginning. Training and validation each get their own loader and token list.

Karpathy's `DataLoaderLite` also loads the next shard when needed. With distributed workers, each rank starts at its own offset and the cursor advances by `B*T*num_processes` so workers consume different portions of the token stream.


In [ ]:
class TinyBatcher:
    def __init__(self, token_ids: list[int], batch_size: int, context_length: int):
        if len(token_ids) < batch_size * context_length + 1:
            raise ValueError("A stream must contain at least B*T+1 token IDs.")
        self.token_ids = token_ids
        self.batch_size = batch_size
        self.context_length = context_length
        self.position = 0

    def next_batch(self) -> tuple[int, list[list[int]], list[list[int]]]:
        start = self.position
        inputs, targets = make_batch(
            self.token_ids, start, self.batch_size, self.context_length
        )
        self.position += self.batch_size * self.context_length
        if self.position + self.batch_size * self.context_length + 1 > len(self.token_ids):
            self.position = 0
        return start, inputs, targets


train_loader = TinyBatcher(train_tokens, BATCH_SIZE, CONTEXT_LENGTH)
validation_loader = TinyBatcher(validation_tokens, BATCH_SIZE, CONTEXT_LENGTH)

for batch_number in range(5):
    start, x, y = train_loader.next_batch()
    print(f"train batch {batch_number + 1}: starts at token {start}; first row x={x[0]}, y={y[0]}; next cursor={train_loader.position}")

val_start, val_x, val_y = validation_loader.next_batch()
print(f"\nvalidation loader starts at its own token 0: x={val_x[0]}, y={val_y[0]}")
assert train_loader.token_ids is not validation_loader.token_ids
assert train_loader.token_ids is train_tokens
assert validation_loader.token_ids is validation_tokens


## Try it

1. Change `BATCH_SIZE` and `CONTEXT_LENGTH`. How many IDs must a stream contain for one batch? Why is there one extra ID?
2. Print the decoded text represented by a row of `x` and the corresponding row of `y`. The target row should be the same stream shifted left by one token.
3. Change the document split, then check that a complete document still belongs to only one split.
4. Compare `TinyBatcher.next_batch` with `DataLoaderLite.next_batch` in the reference. Identify the line that handles the extra target token, the cursor stride, and shard rollover.

**References:** [GPT-2 tokenization](https://github.com/openai/tiktoken), [Karpathy's FineWeb preprocessing](../../build-nanogpt/fineweb.py), [`DataLoaderLite` and train/validation setup](../../build-nanogpt/train_gpt2.py), [PyTorch tensor basics](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html).
